# Studio sistematico comune (frequentista) e veti sul charm

[Giacomo, 2026-10-06] Notebook separato da `Frequentist_Analysis_Bis.ipynb`, per non toccarlo. Stessa lista di varianti e stessa regola dell'analisi bayesiana (`bayes:.../bayesian_nominal.py`):

- **varianti** (ognuna = modello nominale con un solo cambiamento): finestra con estremo basso 5050 e 5150, alto 5450 e 5700; un solo esponenziale; forma del fondo condivisa tra le cariche; rese di fondo simmetriche; segnale Crystal Ball con code libere; $\mu,\sigma$ separati per carica; preselezione ProbK $>0.4$ e $>0.6$, ProbPi $<0.4$ e $<0.6$ (uno alla volta);
- **regola**: per le varianti che cambiano il campione (finestra, preselezione) lo spostamento conta solo se supera $\sigma_B=\sqrt{|\sigma_{var}^2-\sigma_{nom}^2|}$ (Barlow); varianti sullo stesso campione con $\chi^2$ peggiore del nominale di più di 25 sono escluse (respinte dai dati); massimo $|\Delta|$ per gruppo, gruppi in quadratura (`lf.combine_systematics`);
- **veti sul charm** (controllo informativo, fuori dalla sistematica): solo $D^0$, e $D^0+J/\psi+\chi_{c0}$ (riferimento), masse PDG, $\pm30$ MeV per $D^0$ e $J/\psi$, $\pm60$ MeV per $\chi_{c0}$, sulle due coppie $K^+K^-$ di carica opposta.

Percorso dei dati: variabile d'ambiente `LHCB_DATA_DIR` (default il percorso di Marco).

In [1]:
import os, json, warnings
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)
import lhcb_freq as lf

_DATA = Path(os.environ.get("LHCB_DATA_DIR", r"C:\\Users\\marco\\Documents\\Università\\Anno IV\\Statistics_and_Data_Analysis\\Data"))
cfg = lf.Config(data_dir=_DATA, sim_file=_DATA / "PhaseSpaceSimulation.root")
RESULTS_DIR = os.path.join(os.getcwd(), "results"); os.makedirs(RESULTS_DIR, exist_ok=True)

data = lf.load_data(cfg)[cfg.columns]
# campione con tagli piu' larghi di tutte le varianti: scan_config riapplica la preselezione di ogni variante
data_loose = data[lf.kaon_preselection(data, replace(cfg, prob_k_min=0.4, prob_pi_max=0.6))].reset_index(drop=True)
lf.add_B_variables(data_loose, cfg)
nominal_sample = lf.select_window(data_loose[lf.kaon_preselection(data_loose, cfg)], cfg.fit_window)
print("candidati nella finestra nominale:", len(nominal_sample))

/opt/homebrew/Caskroom/miniforge/base/envs/cern/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


candidati nella finestra nominale: 19731


## Varianti

In [2]:
VARIATIONS = [dict(label="nominal"),
    dict(label="window_lo5050", fit_window=(5050.0, 5600.0)), dict(label="window_lo5150", fit_window=(5150.0, 5600.0)),
    dict(label="window_hi5450", fit_window=(5100.0, 5450.0)), dict(label="window_hi5700", fit_window=(5100.0, 5700.0)),
    dict(label="bkg_single_exp", n_exp=1), dict(label="bkg_shared_shape", bkg_shape="shared"),
    dict(label="bkg_symmetric", bkg_yield="symmetric"),
    dict(label="signal_crystal_ball", signal="cb"), dict(label="signal_per_charge", signal_per_charge=True),
    dict(label="presel_probk_0.4", prob_k_min=0.4), dict(label="presel_probk_0.6", prob_k_min=0.6),
    dict(label="presel_probpi_0.4", prob_pi_max=0.4), dict(label="presel_probpi_0.6", prob_pi_max=0.6)]
GROUPS = {"window": ["window_lo5050", "window_lo5150", "window_hi5450", "window_hi5700"],
          "background": ["bkg_single_exp", "bkg_shared_shape", "bkg_symmetric"],
          "signal": ["signal_crystal_ball", "signal_per_charge"],
          "preselection": ["presel_probk_0.4", "presel_probk_0.6", "presel_probpi_0.4", "presel_probpi_0.6"]}
SAMPLE_CHANGING = GROUPS["window"] + GROUPS["preselection"]

res = lf.scan_config(data_loose, cfg, VARIATIONS)
cols = ["label", "valid", "accurate", "n_events", "A_CP", "A_CP_err", "chi2"]
print(res[cols].to_string())
nom = res[res.label == "nominal"].iloc[0]
syst = lf.combine_systematics(nom, [r for _, r in res.iterrows() if r.label != "nominal"], GROUPS, SAMPLE_CHANGING)
for k, v in syst["variations"].items():
    print(f"{k:22s} shift={v['shift']:+.4f}  sigma_B={v['sigma_barlow'] if v['sigma_barlow'] is None else round(v['sigma_barlow'], 4)}"
          f"  {'ESCLUSA (dati)' if v['rejected_by_data'] else ('conta' if v['counts'] else 'non conta (Barlow)')}")
print("per gruppo:", {k: round(v, 4) for k, v in syst["groups"].items()}, " totale:", round(syst["total"], 4),
      " (statistica:", round(syst["sigma_stat"], 4), ")")

scan:   0%|          | 0/14 [00:00<?, ?it/s]

scan:   7%|▋         | 1/14 [00:03<00:47,  3.67s/it]

scan:  14%|█▍        | 2/14 [00:06<00:40,  3.36s/it]

scan:  21%|██▏       | 3/14 [00:08<00:29,  2.72s/it]

scan:  29%|██▊       | 4/14 [00:11<00:28,  2.80s/it]

...

scan:  86%|████████▌ | 12/14 [00:35<00:05,  2.81s/it]

scan:  93%|█████████▎| 13/14 [00:37<00:02,  2.47s/it]

scan: 100%|██████████| 14/14 [00:39<00:00,  2.41s/it]

scan: 100%|██████████| 14/14 [00:39<00:00,  2.81s/it]

                  label  valid  accurate  n_events      A_CP  A_CP_err        chi2
0               nominal   True      True     19731 -0.044129  0.008499  441.716011
1         window_lo5050   True      True     21164 -0.044503  0.008512  519.255452
2         window_lo5150   True     False     18566 -0.044590  0.004306  455.377177
3         window_hi5450   True      True     19004 -0.044222  0.008492  482.733917
4         window_hi5700   True      True     20191 -0.045537  0.008500  473.802492
5        bkg_single_exp   True      True     19731 -0.048060  0.008471  576.264524
6      bkg_shared_shape   True      True     19731 -0.048143  0.008342  451.063731
7         bkg_symmetric   True      True     19731 -0.047238  0.008387  448.433676
8   signal_crystal_ball   True      True     19731 -0.044729  0.008477  406.794596
9     signal_per_charge   True      True     19731 -0.044245  0.008668  441.464469
10     presel_probk_0.4   True      True     26553 -0.047306  0.007640  444.521015
11  

## Veti sul charm (controllo informativo)

In [3]:
vetoes = {}
for name, mode in (("veto_D0", "D"), ("veto_D0_Jpsi_chic0", "all")):
    d = lf.remove_charm_resonances(cfg, nominal_sample, mode)
    m = lf.fit_joint(lf.split_by_charge(d, col="B_Mass"), cfg)
    vetoes[name] = dict(n=int(len(d)), A_CP=float(m.values["A_CP"]), A_CP_err=float(m.errors["A_CP"]),
                        shift=float(m.values["A_CP"] - nom.A_CP), valid=bool(m.valid), accurate=bool(m.accurate))
    print(f"{name:20s} n={len(d)}  A_raw = {m.values['A_CP']:.4f} +- {m.errors['A_CP']:.4f}  (spostamento {m.values['A_CP'] - nom.A_CP:+.4f})")
assert vetoes["veto_D0"]["n"] == 18744 and vetoes["veto_D0_Jpsi_chic0"]["n"] == 16367, "conteggi diversi da quelli concordati"

veto_D0              n=18744  A_raw = -0.0493 +- 0.0087  (spostamento -0.0052)


veto_D0_Jpsi_chic0   n=16367  A_raw = -0.0421 +- 0.0094  (spostamento +0.0021)


## Salvataggio

In [4]:
def _clean(x):
    if isinstance(x, (np.floating, np.integer, np.bool_)):
        return x.item()
    return x
out = {"nominal": {k: _clean(nom[k]) for k in ("A_CP", "A_CP_err", "chi2", "n_events")},
       "variations": [{k: _clean(v) for k, v in r.items() if k != "gof"} for _, r in res.iterrows()],
       "systematics": syst, "charm_vetoes": vetoes, "groups": GROUPS, "sample_changing": SAMPLE_CHANGING}
with open(os.path.join(RESULTS_DIR, "freq_systematics.json"), "w") as f:
    json.dump(out, f, indent=1, default=str)
print("[saved]", os.path.join(RESULTS_DIR, "freq_systematics.json"))

[saved] /Users/gmoretti/Work/wt_freq/LHCb/Code_general/Frequentist/results/freq_systematics.json


## Scansione fine dei tagli PID (controllo di stabilità)

[Giacomo, 2026-10-07] Un taglio alla volta, passo 0.05, per distinguere un andamento in funzione del taglio da una fluttuazione del singolo valore usato nella sistematica (`presel_probk_0.6`). Ogni campione è annidato nel nominale (o lo contiene): la fluttuazione attesa dello spostamento è $\sigma_B=\sqrt{|\sigma_v^2-\sigma_n^2|}$. Non cambia la regola della sistematica.


In [5]:
data_scan = data[lf.kaon_preselection(data, replace(cfg, prob_k_min=0.3, prob_pi_max=0.7))].reset_index(drop=True)
lf.add_B_variables(data_scan, cfg)
k_grid = [round(x, 2) for x in np.arange(0.30, 0.851, 0.05) if abs(x - 0.5) > 1e-9]
pi_grid = [round(x, 2) for x in np.arange(0.15, 0.701, 0.05) if abs(x - 0.5) > 1e-9]
scan_var = [dict(label="nominal")] + [dict(label=f"probk_{k:.2f}", prob_k_min=k) for k in k_grid] \
    + [dict(label=f"probpi_{p:.2f}", prob_pi_max=p) for p in pi_grid]
scan = lf.scan_config(data_scan, cfg, scan_var)
nom_s = scan[scan.label == "nominal"].iloc[0]
rows = []
for _, r in scan.iterrows():
    sb = float(np.sqrt(abs(r.A_CP_err ** 2 - nom_s.A_CP_err ** 2)))
    rows.append({"label": r.label, "prob_k_min": float(r.prob_k_min), "prob_pi_max": float(r.prob_pi_max), "n_events": int(r.n_events),
                 "A_CP": float(r.A_CP), "A_CP_err": float(r.A_CP_err), "shift": float(r.A_CP - nom_s.A_CP), "sigma_barlow": sb,
                 "pull": float((r.A_CP - nom_s.A_CP) / sb) if sb > 0 else 0.0, "valid": bool(r.valid), "accurate": bool(r.accurate)})
    print(f"{r.label:12s} n={int(r.n_events):6d}  A_raw={r.A_CP:+.4f} +- {r.A_CP_err:.4f}  shift={r.A_CP - nom_s.A_CP:+.4f}  sigma_B={sb:.4f}")
with open(os.path.join(RESULTS_DIR, "freq_pid_scan.json"), "w") as f:
    json.dump({"nominal_A": float(nom_s.A_CP), "rows": rows}, f, indent=1)
print("[saved]", os.path.join(RESULTS_DIR, "freq_pid_scan.json"))


scan:   0%|          | 0/23 [00:00<?, ?it/s]

scan:   4%|▍         | 1/23 [00:03<01:12,  3.30s/it]

scan:   9%|▊         | 2/23 [00:10<02:02,  5.84s/it]

scan:  13%|█▎        | 3/23 [00:15<01:46,  5.33s/it]

scan:  17%|█▋        | 4/23 [00:20<01:34,  4.95s/it]

...

scan:  91%|█████████▏| 21/23 [00:52<00:05,  2.50s/it]

scan:  96%|█████████▌| 22/23 [00:54<00:02,  2.41s/it]

scan: 100%|██████████| 23/23 [00:57<00:00,  2.42s/it]

scan: 100%|██████████| 23/23 [00:57<00:00,  2.48s/it]

nominal      n= 19731  A_raw=-0.0441 +- 0.0085  shift=+0.0000  sigma_B=0.0000
probk_0.30   n= 33328  A_raw=-0.0552 +- 0.0072  shift=-0.0111  sigma_B=0.0045
probk_0.35   n= 29279  A_raw=-0.0508 +- 0.0074  shift=-0.0067  sigma_B=0.0041
probk_0.40   n= 26553  A_raw=-0.0473 +- 0.0076  shift=-0.0032  sigma_B=0.0037
probk_0.45   n= 23899  A_raw=-0.0461 +- 0.0079  shift=-0.0019  sigma_B=0.0031
probk_0.55   n= 17112  A_raw=-0.0433 +- 0.0090  shift=+0.0009  sigma_B=0.0030
probk_0.60   n= 13981  A_raw=-0.0384 +- 0.0099  shift=+0.0058  sigma_B=0.0050
probk_0.65   n= 11354  A_raw=-0.0459 +- 0.0108  shift=-0.0017  sigma_B=0.0067
probk_0.70   n=  8649  A_raw=-0.0553 +- 0.0123  shift=-0.0112  sigma_B=0.0089
probk_0.75   n=  6857  A_raw=-0.0495 +- 0.0137  shift=-0.0054  sigma_B=0.0108
probk_0.80   n=  4769  A_raw=-0.0431 +- 0.0163  shift=+0.0010  sigma_B=0.0139
probk_0.85   n=  3322  A_raw=-0.0419 +- 0.0193  shift=+0.0022  sigma_B=0.0174
probpi_0.15  n= 10831  A_raw=-0.0446 +- 0.0111  shift=-0.0004  s